In [5]:
import json
from langchain.docstore.document import Document
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [6]:
with open('data.json','r') as f:
    data = json.load(f)

docs = [Document(page_content=entry['description'], metadata={'title': entry['title'],'leave_type':entry['leave_type']}) for entry in data]

In [7]:
print(docs)

[Document(metadata={'title': 'Sick Leave Policy', 'leave_type': 'Sick Leave'}, page_content='Students may take up to 7 days of sick leave per semester with a medical certificate.'), Document(metadata={'title': 'Casual Leave Policy', 'leave_type': 'Casual Leave'}, page_content='Students are allowed 5 days of casual leave per academic year without needing documentation.'), Document(metadata={'title': 'Medical Emergency Leave', 'leave_type': 'Medical Leave'}, page_content='Extended leave can be granted in case of hospitalization with official medical records.'), Document(metadata={'title': 'Family Emergency Leave', 'leave_type': 'Emergency Leave'}, page_content='Students may request up to 10 days of leave for family emergencies with proper documentation.'), Document(metadata={'title': 'Maternity Leave for Students', 'leave_type': 'Maternity Leave'}, page_content='Pregnant students may take up to 90 days of maternity leave with valid documentation.'), Document(metadata={'title': 'Paternity

In [10]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
     model_name="BAAI/bge-small-en",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

/home/dell/DS Projects/Leave-Policy/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
from langchain.vectorstores import Chroma

vectorestore = Chroma.from_documents(docs,embedding_model)

In [14]:
splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)

In [20]:
store = InMemoryStore()
retriever = ParentDocumentRetriever(
    vectorstore=vectorestore,
    docstore=store,
    child_splitter=splitter,
)

In [23]:
retriever.add_documents(docs)

In [24]:
query = "How many days of sick leave are allowed?"

results = retriever.invoke(query)
print(results[0].metadata['leave_type'])
print(results[0].page_content)

Sick Leave
Students may take up to 7 days of sick leave per semester with a medical certificate.


In [27]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

model = HuggingFaceEndpoint(
    repo_id="meta-llama/Meta-Llama-3-8B-Instruct",
    task="text-generation"
)

llm = ChatHuggingFace(llm=model,temperature=0)

In [32]:
from langchain.prompts import PromptTemplate
prompt_template = """
You are an intelligent assistant designed to answer questions related to an organization's leave policies. Use the provided context from the School/College official leave policy documents to generate accurate, concise, and helpful answers.

Context:
{context}

Question:
{question}

Instructions:
- Only use the information provided in the context to answer.
- If the context does not contain enough information, say: "The document does not contain information about this topic."
- Be specific and avoid vague responses.
- Use bullet points or formatting if it improves clarity.
- Do not make up policies or speculate.

Answer:

"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

In [33]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import  RunnablePassthrough,RunnableSequence

parser = StrOutputParser()

chain = RunnableSequence({"context": retriever, "question": RunnablePassthrough()}, prompt ,llm ,parser)
  
query = "Whats about sick leave?"
response = chain.invoke(query)

print("Answer:", response)

Answer: **Sick Leave Policy:**

According to the Sick Leave Policy document, the following information is available:

* **Maximum Sick Leave per Semester:** Up to 7 days of sick leave per semester.
* **Required Documentation:** A medical certificate is required to support sick leave.

Please note that this information is based solely on the provided context and may not be a comprehensive overview of the organization's leave policies.
